# file_server_clean_up — Safe file audit & cleanup candidates

**What this notebook does**
- Recursively scans a target folder
- Flags obvious junk (OS temp/metadata, zero-byte files, etc.)
- Detects *content duplicates* using hashes (safe, filename-independent)
- Chooses one file to **KEEP** per duplicate set (configurable)
- Exports a **CSV** listing paths and files flagged as **DELETE_CANDIDATE** (no deletion by default)

**Safety**: This notebook does **not** delete anything unless you explicitly enable the optional quarantine step at the end.

## 0) Install checklist (run once)
You should already have created the conda environment and installed dependencies. See the repo README / setup steps from the chat.

In [ ]:
import os
import sys
import math
import time
import hashlib
from dataclasses import dataclass
from pathlib import Path
from datetime import datetime

import pandas as pd
from tqdm import tqdm

print('Python:', sys.version)
print('CWD:', os.getcwd())

## 1) Configuration (edit these)


In [ ]:
# === REQUIRED: set the folder you want to scan ===
# Windows example:
# ROOT = Path(r"D:\\FILE_SERVER_EXPORT")
# Network share example:
# ROOT = Path(r"\\\\SERVER\\Share\\SomeFolder")

ROOT = Path(r"CHANGE_ME")

# Where to write outputs
OUTPUT_DIR = Path("..") / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Exclusions
EXCLUDE_DIR_NAMES = {
    ".git", ".ipynb_checkpoints", "__pycache__", "node_modules",
    "$RECYCLE.BIN", "System Volume Information"
}

# Obvious junk filenames/extensions (add as needed)
JUNK_FILENAMES = {
    ".DS_Store", "Thumbs.db", "desktop.ini", "Icon\r", "._.DS_Store"
}
JUNK_EXTENSIONS = {
    ".tmp", ".temp", ".swp", ".swo", ".bak", ".old", ".dwl", ".dwl2", ".~lock"
}

# Duplicate detection / hashing
HASH_ALGO = "blake2b"   # blake2b is fast & in hashlib
FAST_HASH_BYTES = 2 * 1024 * 1024  # 2 MiB from head + 2 MiB from tail
FULL_HASH_FOR_FINAL_GROUPS = True  # set False to speed up but risk rare collisions

# How to pick the KEEP file within each duplicate group
# Options: "newest_mtime", "oldest_mtime", "shortest_path"
DUP_KEEP_STRATEGY = "newest_mtime"

# Optional directory preferences (higher wins). Example: prefer a 'CURRENT' folder.
# Provide as substrings (case-insensitive) or absolute paths.
PREFERRED_PATH_SUBSTRINGS = [
    # r"\\CURRENT\\",
    # r"\\_LATEST\\",
]

# Minimum file size (bytes) to consider for duplicate grouping
MIN_SIZE_FOR_DUP_CHECK = 1

# CSV name
RUN_TAG = datetime.now().strftime("%Y%m%d_%H%M%S")
CSV_PATH = OUTPUT_DIR / f"cleanup_candidates_{RUN_TAG}.csv"

assert ROOT.exists(), f"ROOT does not exist: {ROOT}"

## 2) File scan (metadata only)


In [ ]:
def iter_files(root: Path):
    # Conservative walk: ignore symlinks by default
    for dirpath, dirnames, filenames in os.walk(root):
        # prune excluded directories in-place
        dirnames[:] = [d for d in dirnames if d not in EXCLUDE_DIR_NAMES]
        for fn in filenames:
            p = Path(dirpath) / fn
            try:
                if p.is_symlink():
                    continue
                st = p.stat()
            except (FileNotFoundError, PermissionError, OSError):
                continue
            yield {
                "path": str(p),
                "name": p.name,
                "ext": p.suffix.lower(),
                "size": int(st.st_size),
                "mtime": float(st.st_mtime),
            }

rows = []
for rec in tqdm(iter_files(ROOT), desc="Scanning", unit="files"):
    rows.append(rec)

df = pd.DataFrame(rows)
df["mtime_iso"] = pd.to_datetime(df["mtime"], unit="s", utc=True).dt.tz_convert(None).astype(str)
df["is_zero_bytes"] = df["size"] == 0
df["is_junk_name"] = df["name"].isin(JUNK_FILENAMES)
df["is_junk_ext"] = df["ext"].isin(JUNK_EXTENSIONS)
df["junk_reason"] = ""
df.loc[df["is_zero_bytes"], "junk_reason"] += "zero_bytes;"
df.loc[df["is_junk_name"], "junk_reason"] += "junk_filename;"
df.loc[df["is_junk_ext"], "junk_reason"] += "junk_extension;"

df.shape, df.head()

## 3) Hashing helpers


In [ ]:
def _new_hasher(algo: str):
    if algo == "blake2b":
        return hashlib.blake2b(digest_size=32)
    if algo == "sha256":
        return hashlib.sha256()
    raise ValueError(f"Unsupported algo: {algo}")

def fast_fingerprint(path: str, size: int, algo: str = HASH_ALGO, nbytes: int = FAST_HASH_BYTES) -> str:
    """Fast fingerprint: hash(first nbytes + last nbytes + size)."""
    h = _new_hasher(algo)
    h.update(str(size).encode("utf-8"))
    p = Path(path)
    with p.open("rb") as f:
        head = f.read(nbytes)
        h.update(head)
        if size > nbytes:
            # tail
            try:
                f.seek(max(0, size - nbytes))
                tail = f.read(nbytes)
                h.update(tail)
            except OSError:
                # some special files may not seek cleanly
                pass
    return h.hexdigest()

def full_hash(path: str, algo: str = HASH_ALGO, chunk_size: int = 8 * 1024 * 1024) -> str:
    h = _new_hasher(algo)
    p = Path(path)
    with p.open("rb") as f:
        while True:
            b = f.read(chunk_size)
            if not b:
                break
            h.update(b)
    return h.hexdigest()

## 4) Duplicate detection (size → fast hash → full hash)


In [ ]:
df["dup_eligible"] = df["size"] >= MIN_SIZE_FOR_DUP_CHECK

# Pass 1: only consider size groups with >1 files
size_counts = df.loc[df["dup_eligible"], "size"].value_counts()
candidate_sizes = set(size_counts[size_counts > 1].index.tolist())

cand = df[df["dup_eligible"] & df["size"].isin(candidate_sizes)].copy()
print("Duplicate candidates by size:", len(cand), "files across", len(candidate_sizes), "sizes")

# Pass 2: compute fast fingerprints for candidates
fast_hashes = []
for path, size in tqdm(zip(cand["path"], cand["size"]), total=len(cand), desc="Fast hashing"):
    try:
        fast_hashes.append(fast_fingerprint(path, int(size)))
    except (FileNotFoundError, PermissionError, OSError):
        fast_hashes.append(None)

cand["hash_fast"] = fast_hashes
cand = cand.dropna(subset=["hash_fast"]).copy()

# Pass 3: group by (size, hash_fast), and optionally full-hash within groups >1
cand["group_fast"] = cand.groupby(["size", "hash_fast"]).ngroup()
fast_group_sizes = cand["group_fast"].value_counts()
fast_dupe_groups = fast_group_sizes[fast_group_sizes > 1].index.tolist()

cand["hash_full"] = None
if FULL_HASH_FOR_FINAL_GROUPS and fast_dupe_groups:
    mask = cand["group_fast"].isin(fast_dupe_groups)
    sub = cand.loc[mask].copy()
    fulls = []
    for path in tqdm(sub["path"], total=len(sub), desc="Full hashing"):
        try:
            fulls.append(full_hash(path))
        except (FileNotFoundError, PermissionError, OSError):
            fulls.append(None)
    cand.loc[mask, "hash_full"] = fulls

# Determine the final duplicate key
cand["dup_key"] = cand.apply(
    lambda r: (r["size"], r["hash_full"]) if (FULL_HASH_FOR_FINAL_GROUPS and pd.notna(r["hash_full"])) else (r["size"], r["hash_fast"]),
    axis=1,
)

cand["dup_group_id"] = cand.groupby("dup_key").ngroup()
dup_group_counts = cand["dup_group_id"].value_counts()
dup_groups = dup_group_counts[dup_group_counts > 1].index.tolist()

print("Final duplicate groups:", len(dup_groups))
dup_group_counts.head(10)

## 5) Decide KEEP vs DELETE_CANDIDATE


In [ ]:
def preference_score(path: str) -> int:
    p = path.lower()
    score = 0
    for i, sub in enumerate(PREFERRED_PATH_SUBSTRINGS):
        if sub and sub.lower() in p:
            # earlier in list = higher weight
            score += (len(PREFERRED_PATH_SUBSTRINGS) - i) * 10_000
    # shorter path gets a tiny bump (helps tie-break)
    score += max(0, 500 - len(path))
    return score

cand["pref_score"] = cand["path"].apply(preference_score)

def pick_keeper(group: pd.DataFrame) -> int:
    if DUP_KEEP_STRATEGY == "newest_mtime":
        # preference score dominates, then newest mtime
        g = group.sort_values(["pref_score", "mtime"], ascending=[False, False])
    elif DUP_KEEP_STRATEGY == "oldest_mtime":
        g = group.sort_values(["pref_score", "mtime"], ascending=[False, True])
    elif DUP_KEEP_STRATEGY == "shortest_path":
        g = group.assign(path_len=group["path"].str.len()).sort_values(["pref_score", "path_len"], ascending=[False, True])
    else:
        raise ValueError(f"Unknown strategy: {DUP_KEEP_STRATEGY}")
    return g.index[0]

# Mark keepers within each duplicate group
cand["dup_is_keeper"] = False
keepers = []
for gid, g in tqdm(cand[cand["dup_group_id"].isin(dup_groups)].groupby("dup_group_id"), desc="Choosing keepers"):
    keep_idx = pick_keeper(g)
    keepers.append(keep_idx)
cand.loc[keepers, "dup_is_keeper"] = True

# Merge duplicate info back to full df
df = df.merge(
    cand[["path", "hash_fast", "hash_full", "dup_group_id", "dup_is_keeper"]],
    on="path", how="left"
)

# Decision logic (conservative):
# - If junk => DELETE_CANDIDATE
# - Else if in duplicate group and not keeper => DELETE_CANDIDATE
# - Else => KEEP
df["decision"] = "KEEP"
df["reason"] = "";
df.loc[df["junk_reason"] != "", "decision"] = "DELETE_CANDIDATE"
df.loc[df["junk_reason"] != "", "reason"] += df.loc[df["junk_reason"] != "", "junk_reason"]

dup_nonkeepers = df["dup_group_id"].notna() & (df["dup_is_keeper"] == False)
df.loc[dup_nonkeepers & (df["decision"] == "KEEP"), "decision"] = "DELETE_CANDIDATE"
df.loc[dup_nonkeepers & (df["decision"] == "DELETE_CANDIDATE"), "reason"] += "duplicate_nonkeeper;"
df.loc[dup_nonkeepers & (df["decision"] == "KEEP"), "reason"] += "duplicate_nonkeeper;"

# For keepers in a dupe group, note it
df.loc[df["dup_group_id"].notna() & (df["dup_is_keeper"] == True), "reason"] += "duplicate_keeper;"

# Clean reason formatting
df["reason"] = df["reason"].str.strip(";")

df["decision"].value_counts(), df.head()

## 6) Export CSV (paths + flags)


In [ ]:
out = df[[
    "path", "decision", "reason", "size", "mtime_iso",
    "ext", "hash_fast", "hash_full", "dup_group_id", "dup_is_keeper"
]].copy()

out.to_csv(CSV_PATH, index=False, encoding="utf-8")
print("Wrote:", CSV_PATH)
out["decision"].value_counts()

## 7) Review helpers


In [ ]:
# Preview what would be deleted
out[out["decision"] == "DELETE_CANDIDATE"].head(50)

In [ ]:
# Counts by reason (for DELETE_CANDIDATE rows)
tmp = out[out['decision'] == 'DELETE_CANDIDATE'].copy()
tmp['reason_list'] = tmp['reason'].fillna('').apply(lambda s: [r for r in s.split(';') if r])
reason_counts = (
    tmp.explode('reason_list')
       .query("reason_list != ''")
       .groupby('reason_list')
       .size()
       .sort_values(ascending=False)
)
reason_counts.head(50)


In [ ]:
# Largest delete candidates (often duplicates)
out[out["decision"] == "DELETE_CANDIDATE"].sort_values("size", ascending=False).head(50)

## 8) OPTIONAL: Quarantine move (safer than delete)
**Default is disabled.** If you enable it, files are moved to a quarantine folder preserving relative paths.

⚠️ Only do this after reviewing the CSV.


In [ ]:
ENABLE_QUARANTINE = False
QUARANTINE_DIR = Path("..") / "quarantine" / f"run_{RUN_TAG}"

def move_to_quarantine(src_path: str, root: Path, quarantine_root: Path):
    src = Path(src_path)
    rel = src.relative_to(root)
    dst = quarantine_root / rel
    dst.parent.mkdir(parents=True, exist_ok=True)
    src.replace(dst)  # atomic move on same volume

if ENABLE_QUARANTINE:
    QUARANTINE_DIR.mkdir(parents=True, exist_ok=True)
    to_move = out[out["decision"] == "DELETE_CANDIDATE"]["path"].tolist()
    moved = 0
    for p in tqdm(to_move, desc="Quarantining"):
        try:
            move_to_quarantine(p, ROOT, QUARANTINE_DIR)
            moved += 1
        except Exception as e:
            print("FAILED:", p, "->", e)
    print(f"Moved {moved}/{len(to_move)} files to {QUARANTINE_DIR}")
else:
    print("Quarantine is disabled (ENABLE_QUARANTINE=False).")